# Curiosity-Driven Mode Switching — Detection Module
**Goal.** A fast end-to-end (E2E) driving policy runs by default; detect *when it is about
to fail* so control can escalate to a slower, stronger VLA. This notebook is the **detection
module**: a passive failure monitor.

**Method (how to read every result below).**
- **Failure label** = *measured* trajectory error (ADE) in the worst 10% of frames. It is a
  measurement, not an opinion. **ADE is never an input feature** (anti-circular).
- **Candidate signals**: cross-model *disagreement*, single-model *native uncertainty*,
  VLM *perceived risk*, and *temporal accumulation*.
- **Evaluation**: 5-fold **scene-level** cross-validation (no scene split across folds);
  the VLM ensemble is **zero-shot** (no training-scene leakage).

This notebook only **loads saved results** (heavy inference is already done) and re-renders
them, plus one quick live recompute of the core transfer result.

In [1]:
import pandas as pd, numpy as np
REPORT='results/curiosity/report'; TRAJ='results/curiosity/traj'; CUR='results/curiosity'
pd.set_option('display.max_colwidth', 60)
print('loaded libs; reading artifacts from', REPORT)
print('(figures are saved as PNG under results/curiosity/report/ — this notebook shows the numeric results)')

loaded libs; reading artifacts from results/curiosity/report
(figures are saved as PNG under results/curiosity/report/ — this notebook shows the numeric results)


## 1. Which signal predicts failure?
Single-signal detection AUROC/AUPRC (chance AUROC 0.50; baseline AUPRC = positive rate 0.10).

In [2]:
sig = pd.read_csv(f'{REPORT}/comparison_table.csv'); display(sig)

,signal,set,n,label_def,AUROC,AUPRC
0,VLM-ensemble disagreement,"nuScenes (DriveLM, 150sc/3908fr)",3908,VLM-mean ADE top10%,0.798,0.375
1,SparseDrive native (margin),nuScenes val (150sc/6019fr),6019,SD ADE top10%,0.551,0.107
2,SparseDrive native (entropy),nuScenes val (150sc/6019fr),6019,SD ADE top10%,0.546,0.114
3,SparseDrive mode-spread (intra-disagreement),nuScenes val (150sc/6019fr),6019,SD ADE top10%,0.528,0.128
4,DiffusionDrive native (margin),nuScenes val (150sc/6019fr),6019,DD ADE top10%,0.606,0.136
5,DiffusionDrive mode-spread,nuScenes val (150sc/6019fr),6019,DD ADE top10%,0.506,0.100
6,VAD-Base native,nuScenes val,-,NaN,NaN,NaN
7,Driving-model disagreement (6 models),Waymo anchor479 (473fr),473,rap ADE top10%,0.584,0.153
8,VLM teacher risk (escalation),Waymo anchor479 (473fr),473,rap ADE top10%,0.457,0.103


**Takeaway.** Cross-model disagreement is the strongest signal (VLM ensemble **0.80**); single-model native uncertainty is weak (0.55–0.61); VLM perceived risk is ≈ chance (0.46).

## 2. Learned detector (Failure Detection module)
DriveLM-nuScenes, 5-fold scene CV. Three models of increasing expressiveness.
(Architecture diagram: `results/curiosity/report/architecture.png`.)

In [3]:
dl = pd.read_csv(f'{REPORT}/detector_dl.csv'); display(dl)
print('LR / MLP / DeepSets all reach AUROC ~0.82')

,model,AUROC_mean,AUROC_std,AUPRC
0,LR,0.815,0.034,0.410
1,MLP,0.812,0.037,0.439
2,SetNet,0.819,0.023,0.402


LR / MLP / DeepSets all reach AUROC ~0.82


**Takeaway.** LR (0.815), MLP (0.812) and a permutation-invariant **DeepSets** (0.819) are indistinguishable; DeepSets learns the disagreement representation directly from raw trajectories.

## 3. Feature importance (which disagreement statistic drives it)

In [4]:
imp = pd.read_csv(f'{REPORT}/detector_drivelm.csv', skip_blank_lines=False)
fi = imp[imp['detector'].astype(str).str.contains('feature_importance', na=False)].index
rows = imp.iloc[fi[0]+1:].dropna(subset=['detector']) if len(fi) else imp.iloc[0:0]
rows = rows[['detector','features']].rename(columns={'detector':'feature','features':'coef'})
rows['coef']=rows['coef'].astype(float)
display(rows.sort_values('coef', key=abs, ascending=False))

,feature,coef
4,pathlen_std,1.374
5,disp_1s,0.983
6,disp_3s,-0.794
7,endpoint_lat_std,0.780
8,max_pair_endpoint,-0.201
9,disp_mean,-0.170
10,heading_std,0.113
11,n_models,0.003


**Takeaway.** Near-term, speed-related disagreement dominates: `pathlen_std` (+1.37) and 1 s spread (+0.98); long-horizon (3 s) spread is not predictive.

## 4. Single-model internal signals only (contrast)
A detector limited to one model's own signals — the only option without an ensemble.

In [5]:
display(pd.read_csv(f'{REPORT}/detector_table.csv'))
print('Internal-signal fused ~0.61  vs  cross-model disagreement ~0.82')

,set,detector,features,AUROC,AUPRC,note
0,SparseDrive,disagreement,sig_mode_var+sig_mode_std,0.456±0.050,0.107±0.033,NaN
1,SparseDrive,native,sig_entropy+sig_margin,0.554±0.063,0.118±0.025,NaN
2,SparseDrive,fused,sig_mode_var+sig_mode_std+sig_entropy+sig_margin,0.614±0.028,0.166±0.051,NaN
3,DiffusionDrive,disagreement,sig_mode_var+sig_mode_std,0.516±0.072,0.116±0.024,NaN
4,DiffusionDrive,native,sig_entropy+sig_margin,0.610±0.030,0.184±0.034,NaN
5,DiffusionDrive,fused,sig_mode_var+sig_mode_std+sig_entropy+sig_margin,0.603±0.034,0.180±0.036,NaN
6,VAD-Base,-,-,NaN,NaN,single-mode: no native/disagreement signal


Internal-signal fused ~0.61  vs  cross-model disagreement ~0.82


## 5. Robustness to the failure threshold (5/10/20%)

In [6]:
display(pd.read_csv(f'{REPORT}/robustness_table.csv'))
print('Conclusion stable across thresholds (all above baseline).')

,set,detector,fail_pct,AUROC,AUPRC
0,SparseDrive,fused,5,0.576±0.049,0.070±0.014
1,SparseDrive,fused,10,0.614±0.028,0.166±0.051
2,SparseDrive,fused,20,0.619±0.027,0.284±0.047
3,DiffusionDrive,fused,5,0.618±0.031,0.099±0.015
4,DiffusionDrive,fused,10,0.603±0.034,0.180±0.036
5,DiffusionDrive,fused,20,0.613±0.023,0.295±0.028


Conclusion stable across thresholds (all above baseline).


## 6. Transfer test (core result)
Does cross-model disagreement predict an **actual driving model's** failure?
Official nuScenes-val, 6019 frames, label = **SparseDrive** ADE top-10%.

In [7]:
display(pd.read_csv(f'{REPORT}/transfer_fusion.csv'))

,detector,AUROC,AUROC_std,AUPRC,AUPRC_std
0,native-only,0.618,0.026,0.170,0.058
1,disagreement-only,0.755,0.018,0.290,0.055
2,fused,0.762,0.023,0.293,0.054
3,leave-SD-out(DD vs VAD only),0.667,0.022,0.191,0.000


### Live recompute (from the shared trajectory + signal CSVs)
Rebuilds the transfer detector from `traj/{sd,dd,vad}_trajs.csv` + `sd_signals.csv` with
5-fold scene CV — should reproduce ≈ native 0.62 / disagreement 0.76 / fused 0.76.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def load_traj(p):
    df=pd.read_csv(p); cols=[f't{i}{a}' for i in range(6) for a in 'xy']
    return {t:df[df.token==t][cols].values.reshape(6,2) for t in df.token} if False else            {r.token:r[cols].values.astype(float).reshape(6,2) for _,r in df.iterrows()}
sd=load_traj(f'{TRAJ}/sd_trajs.csv'); dd=load_traj(f'{TRAJ}/dd_trajs.csv'); vad=load_traj(f'{TRAJ}/vad_trajs.csv')
sg=pd.read_csv(f'{CUR}/sd_signals.csv').set_index('token')
toks=[t for t in sg.index if t in sd and t in dd and t in vad]
DIS,NAT,y,scn=[],[],[],[]
for t in toks:
    T=np.stack([sd[t],dd[t],vad[t]]); pw=np.mean(np.sum((T-T.mean(0))**2,axis=2),axis=0); ep=T[:,-1]
    seg=np.linalg.norm(np.diff(T,axis=1),axis=2).sum(1)
    mp=max(np.linalg.norm(ep[a]-ep[b]) for a in range(3) for b in range(a+1,3))
    DIS.append([pw.mean()**.5,pw[1]**.5,pw[-1]**.5,np.std(ep[:,0]),np.std(seg),mp,
                float(np.mean(np.linalg.norm(dd[t]-vad[t],axis=1)))])
    NAT.append([sg.loc[t,'sig_margin'],sg.loc[t,'sig_entropy'],sg.loc[t,'sig_mode_std']])
    y.append(int(sg.loc[t,'fail_top10'])); scn.append(sg.loc[t,'scene'])
DIS,NAT,y,scn=np.array(DIS),np.array(NAT),np.array(y),np.array(scn)
def cv(X):
    a=[]
    for tr,te in GroupKFold(5).split(X,y,scn):
        s=StandardScaler().fit(X[tr]); m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(s.transform(X[tr]),y[tr])
        a.append(roc_auc_score(y[te],m.predict_proba(s.transform(X[te]))[:,1]))
    return np.mean(a)
print(f'n={len(toks)} frames | positives={y.mean():.1%}')
print(f'native-only       AUROC {cv(NAT):.3f}')
print(f'disagreement-only AUROC {cv(DIS):.3f}')
print(f'fused             AUROC {cv(np.hstack([DIS,NAT])):.3f}')

n=6019 frames | positives=10.0%
native-only       AUROC 0.618


disagreement-only AUROC 0.755


fused             AUROC 0.762


**Takeaway.** Disagreement predicts the driving planner's failure at **0.76**, far above native (0.62). Excluding the target model (leave-SD-out, DD vs VAD only) still gives **0.667** > native — so the effect is **not self-referential**. (Conservative: only 3 driving models; the diverse 14-VLM ensemble was not re-run on nuScenes-val — paid APIs.)

## 7. Negative results

In [9]:
print('VLM perceived risk vs actual failure : AUROC 0.46  (Spearman -0.03 .. +0.07)  -> OOD != failure')
print('Temporal accumulation (open-loop)     : single-frame 0.798 > EWMA 0.760 > CUSUM 0.715  -> no benefit')
# (values from results/curiosity/report/RESULTS.md)

VLM perceived risk vs actual failure : AUROC 0.46  (Spearman -0.03 .. +0.07)  -> OOD != failure
Temporal accumulation (open-loop)     : single-frame 0.798 > EWMA 0.760 > CUSUM 0.715  -> no benefit


## 8. Conclusion
- **Cross-model disagreement** (computational *ambiguity*) is the most informative failure
  signal — 0.80 for a VLM ensemble, and it **transfers** to an actual driving planner (**0.76**,
  vs 0.62 for the planner's own uncertainty; 0.67 even with the target model excluded).
- A small **learned detector** (LR ≈ MLP ≈ DeepSets ≈ **0.82**) captures it; single-model
  internal signals alone reach only ≈0.61.
- **Negative**: VLM perceived risk ≠ failure; temporal accumulation gives no open-loop benefit.
- **Next step**: *distill* the (expensive) disagreement signal into a single-pass trigger and
  evaluate it in closed loop.